In [65]:
import numpy as np
import matplotlib.pyplot as plt  
from sklearn.preprocessing import MinMaxScaler, StandardScaler

In [66]:
def train_test_split(X, y, treino=0.8, seed=42):
    if seed is not None:
        np.random.seed(seed)  

    n = X.shape[0]

    indices = np.random.permutation(n)

    split = int(n * treino)

    train_idx = indices[:split]
    test_idx = indices[split:]

    X_train = X[train_idx]
    y_train = y[train_idx]

    X_test = X[test_idx]
    y_test = y[test_idx]

    return X_train, X_test, y_train, y_test

# Questão 1

In [67]:
breastcance = np.loadtxt("breastcancer.csv", delimiter=",", skiprows=1)

In [68]:
X = breastcance[:, :30]

y = breastcance[:, 30]

In [69]:
X_train, X_test, y_train, y_test = train_test_split(X, y, treino=0.8, seed=42)   

In [ ]:
def k_fold_cross_validation(model_class, X, y, k=10, seed=42, **model_params):
    n = X.shape[0]
    indices = np.arange(n)
    np.random.seed(seed)
    np.random.shuffle(indices)

    fold_size = n // k

    global_accuracies = []
    class_accuracies = []

    classes = np.unique(y)

    for i in range(k):
        start = i * fold_size
        end = start + fold_size

        test_idx = indices[start:end]
        train_idx = np.concatenate((indices[:start], indices[end:]))

        X_train, y_train = X[train_idx], y[train_idx]
        X_test, y_test = X[test_idx], y[test_idx]

        model = model_class(**model_params)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        acc_global = np.mean(y_pred.flatten() == y_test.flatten())
        global_accuracies.append(acc_global)

        acc_per_class = []
        for c in classes:
            idx = (y_test == c)
            if np.sum(idx) == 0:
                acc_per_class.append(0)
            else:
                acc_c = np.mean(y_pred[idx] == y_test[idx])
                acc_per_class.append(acc_c)

        class_accuracies.append(acc_per_class)

        print(f"Fold {i+1}")
        print(f"  Global Accuracy: {acc_global:.4f}")
        for j, c in enumerate(classes):
            print(f"  Class {c} Accuracy: {acc_per_class[j]:.4f}")
        print("-" * 30)

    class_accuracies = np.array(class_accuracies)

    print(f"Global Accuracy -> Média: {np.mean(global_accuracies):.4f} | Desvio: {np.std(global_accuracies):.4f}")

    for j, c in enumerate(classes):
        mean_c = np.mean(class_accuracies[:, j])
        std_c = np.std(class_accuracies[:, j])
        print(f"Classe {c} -> Média: {mean_c:.4f} | Desvio: {std_c:.4f}")

    return global_accuracies, class_accuracies

### Letra A

#### Regressão logística Binária

In [71]:
class LogisticRegressionBinary:
    def __init__(self, lr=0.1, epochs=1000):
        self.lr = lr
        self.epochs = epochs

    def __sigmoid(self, z):
        z = np.clip(z, -500, 500)  
        return 1 / (1 + np.exp(-z))

    def __compute_loss(self, y_true, y_pred):
        m = y_true.shape[0]
        return -np.sum(
            y_true * np.log(y_pred + 1e-9) +
            (1 - y_true) * np.log(1 - y_pred + 1e-9)
        ) / m

    def fit(self, X, y):
        m, n = X.shape
        y = y.reshape(-1, 1)

        self.W = np.zeros((n, 1))
        self.b = np.zeros((1, 1))

        for epoch in range(self.epochs):
            z = X @ self.W + self.b
            y_pred = self.__sigmoid(z)

            loss = self.__compute_loss(y, y_pred)

            dz = (y_pred - y) / m
            dW = X.T @ dz
            db = np.sum(dz, axis=0, keepdims=True)

            self.W -= self.lr * dW
            self.b -= self.lr * db

            if epoch % 100 == 0:
                print(f"Epoch {epoch} - Loss: {loss:.4f}")

    def predict(self, X):
        z = X @ self.W + self.b
        y_pred = self.__sigmoid(z)
        return (y_pred >= 0.5).astype(int)

    def score(self, X, y):
        y = y.reshape(-1, 1)
        y_pred = self.predict(X)
        return np.mean(y_pred == y)

In [72]:
lrb = LogisticRegressionBinary(lr=0.1, epochs=1000)
global_accuracies, class_accuracies = k_fold_cross_validation(LogisticRegressionBinary, X, y, k=10, lr=0.1, epochs=1000)

Epoch 0 - Loss: 0.6931
Epoch 100 - Loss: 6.7189
Epoch 200 - Loss: 2.0642
Epoch 300 - Loss: 2.1047
Epoch 400 - Loss: 2.0642
Epoch 500 - Loss: 1.7809
Epoch 600 - Loss: 1.7809
Epoch 700 - Loss: 6.3951
Epoch 800 - Loss: 1.8214
Epoch 900 - Loss: 1.7025
Fold 1
  Global Accuracy: 0.9464
  Class 0.0 Accuracy: 0.9459
  Class 1.0 Accuracy: 0.9474
------------------------------
Epoch 0 - Loss: 0.6931
Epoch 100 - Loss: 13.1139
Epoch 200 - Loss: 2.2261
Epoch 300 - Loss: 2.1452
Epoch 400 - Loss: 1.9023
Epoch 500 - Loss: 1.8347
Epoch 600 - Loss: 1.8493
Epoch 700 - Loss: 1.7000
Epoch 800 - Loss: 1.7373
Epoch 900 - Loss: 1.5787
Fold 2
  Global Accuracy: 0.8750
  Class 0.0 Accuracy: 0.9655
  Class 1.0 Accuracy: 0.7778
------------------------------
Epoch 0 - Loss: 0.6931
Epoch 100 - Loss: 12.8306
Epoch 200 - Loss: 1.9428
Epoch 300 - Loss: 2.2261
Epoch 400 - Loss: 2.1857
Epoch 500 - Loss: 2.0766
Epoch 600 - Loss: 2.2665
Epoch 700 - Loss: 1.7809
Epoch 800 - Loss: 1.7404
Epoch 900 - Loss: 1.7404
Fold 3
  G

In [73]:
np.mean(global_accuracies), np.std(global_accuracies)

(np.float64(0.8803571428571428), np.float64(0.07907710651958758))

In [74]:
for j, c in enumerate(np.unique(y)):
    mean_c = np.mean(class_accuracies[:, j])
    std_c = np.std(class_accuracies[:, j])
    print(f"Classe {c} -> Média: {mean_c:.4f} | Desvio: {std_c:.4f}")

Classe 0.0 -> Média: 0.9298 | Desvio: 0.0678
Classe 1.0 -> Média: 0.8104 | Desvio: 0.2188


#### Análise do discriminante Gaussiano

#### Naive Bayes Gaussiano

### Letra B

# Questão 2

In [75]:
vehicle= np.loadtxt("vehicle.csv", delimiter=",", skiprows=1)

In [76]:
X = vehicle[:, :18]

y = vehicle[:, 18]

In [77]:
X_train, X_test, y_train, y_test = train_test_split(X, y, treino=0.8, seed=42)   

### Letra A

#### Regressão logística Binária

In [78]:
lrs = LogisticRegressionBinary(lr=0.1, epochs=1000)
global_accuracies, class_accuracies = k_fold_cross_validation(LogisticRegressionBinary, X, y, k=10, lr=0.1, epochs=1000)

Epoch 0 - Loss: 0.6931
Epoch 100 - Loss: -9.7762
Epoch 200 - Loss: -9.7762
Epoch 300 - Loss: -9.7762
Epoch 400 - Loss: -9.7762
Epoch 500 - Loss: -9.7762
Epoch 600 - Loss: -9.7762
Epoch 700 - Loss: -9.7762
Epoch 800 - Loss: -9.7762
Epoch 900 - Loss: -9.7762
Fold 1
  Global Accuracy: 0.2619
  Class 0.0 Accuracy: 0.0000
  Class 1.0 Accuracy: 1.0000
  Class 2.0 Accuracy: 0.0000
  Class 3.0 Accuracy: 0.0000
------------------------------
Epoch 0 - Loss: 0.6931
Epoch 100 - Loss: -9.3949
Epoch 200 - Loss: -9.3949
Epoch 300 - Loss: -9.3949
Epoch 400 - Loss: -9.3949
Epoch 500 - Loss: -9.3949
Epoch 600 - Loss: -9.3949
Epoch 700 - Loss: -9.3949
Epoch 800 - Loss: -9.3949
Epoch 900 - Loss: -9.3949
Fold 2
  Global Accuracy: 0.2024
  Class 0.0 Accuracy: 0.0000
  Class 1.0 Accuracy: 1.0000
  Class 2.0 Accuracy: 0.0000
  Class 3.0 Accuracy: 0.0000
------------------------------
Epoch 0 - Loss: 0.6931
Epoch 100 - Loss: -9.8851
Epoch 200 - Loss: -9.8851
Epoch 300 - Loss: -9.8851
Epoch 400 - Loss: -9.8851

In [79]:
np.mean(global_accuracies), np.std(global_accuracies)

(np.float64(0.25), np.float64(0.035315230890931734))

In [80]:
for j, c in enumerate(np.unique(y)):
    mean_c = np.mean(class_accuracies[:, j])
    std_c = np.std(class_accuracies[:, j])
    print(f"Classe {c} -> Média: {mean_c:.4f} | Desvio: {std_c:.4f}")

Classe 0.0 -> Média: 0.0000 | Desvio: 0.0000
Classe 1.0 -> Média: 1.0000 | Desvio: 0.0000
Classe 2.0 -> Média: 0.0000 | Desvio: 0.0000
Classe 3.0 -> Média: 0.0000 | Desvio: 0.0000


#### Análise do discriminante Gaussiano

#### Naive Bayes Gaussiano

### Letra B